# IS 507 Data Preparation and Profiling

This notebook uses the two single-file Parquet datasets shared with the team:

- `data/asi_opensource_pod_hourly_day0_29.parquet`
- `data/asi_opensource_job_execution_summary.parquet`

The workflow is intentionally simple:

1. Validate file completeness and inspect missing or exceptional values.
2. Define transparent query-time adjustments without rewriting the raw files.
3. Produce compact field, numeric, and categorical profile tables.

The original 720 hourly files may remain on the data owner's machine, but this notebook never requires them.


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "MID" / "data").exists():
    ROOT = ROOT / "MID"

DATA_DIR = ROOT / "data"
POD_PARQUET = DATA_DIR / "asi_opensource_pod_hourly_day0_29.parquet"
JOB_PARQUET = DATA_DIR / "asi_opensource_job_execution_summary.parquet"

assert POD_PARQUET.exists(), f"Missing pod-hourly file: {POD_PARQUET}"
assert JOB_PARQUET.exists(), f"Missing execution-summary file: {JOB_PARQUET}"

con = duckdb.connect()
con.execute("SET threads=4")
con.execute("SET memory_limit='8GB'")
TEMP_DIR = DATA_DIR / '.duckdb_tmp'
TEMP_DIR.mkdir(parents=True, exist_ok=True)
con.execute(f"SET temp_directory='{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order=false")

con.execute(f'''CREATE OR REPLACE VIEW pod_raw AS
                SELECT * FROM read_parquet('{POD_PARQUET.as_posix()}')''')
con.execute(f'''CREATE OR REPLACE VIEW job_raw AS
                SELECT * FROM read_parquet('{JOB_PARQUET.as_posix()}')''')

# Day 0 keeps interactive profiling fast. Add days only when a full scan is needed.
POD_PROFILE_DAYS = [0]
profile_days_sql = ', '.join(map(str, POD_PROFILE_DAYS))
con.execute(f'''CREATE OR REPLACE VIEW pod_profile_raw AS
                SELECT * FROM pod_raw WHERE day IN ({profile_days_sql})''')

print("Pod-hourly:", POD_PARQUET)
print("Execution summary:", JOB_PARQUET)


## 1. Completeness and missing-value analysis

The first checks use Parquet metadata and the physical `day` and `hour` columns. Expected row counts are fixed for this project copy so that an incomplete or incorrect shared file fails visibly. Missing-value and profile queries use `POD_PROFILE_DAYS = [0]` by default so they remain interactive; change the list only when a full multi-day profile is required.


In [ ]:
EXPECTED_POD_ROWS = 842_390_418
EXPECTED_JOB_ROWS = 40_522_321

def parquet_metadata(path):
    parquet = pq.ParquetFile(path)
    return {
        "file": path.name,
        "size_gib": path.stat().st_size / 1024**3,
        "rows": parquet.metadata.num_rows,
        "row_groups": parquet.metadata.num_row_groups,
        "columns": parquet.metadata.num_columns,
    }

file_inventory = pd.DataFrame([
    parquet_metadata(POD_PARQUET),
    parquet_metadata(JOB_PARQUET),
])
display(file_inventory)

assert int(file_inventory.loc[0, "rows"]) == EXPECTED_POD_ROWS
assert int(file_inventory.loc[1, "rows"]) == EXPECTED_JOB_ROWS

pod_coverage = con.execute('''
SELECT
    count(*) AS rows,
    min(day) AS min_day,
    max(day) AS max_day,
    count(DISTINCT day) AS distinct_days,
    min(hour) AS min_hour,
    max(hour) AS max_hour,
    count(DISTINCT (day, hour)) AS distinct_day_hours
FROM pod_raw
''').df()
display(pod_coverage)

coverage = pod_coverage.iloc[0]
assert int(coverage["rows"]) == EXPECTED_POD_ROWS
assert (int(coverage["min_day"]), int(coverage["max_day"])) == (0, 29)
assert int(coverage["distinct_days"]) == 30
assert (int(coverage["min_hour"]), int(coverage["max_hour"])) == (0, 23)
assert int(coverage["distinct_day_hours"]) == 720
print("Completeness checks passed.")


In [ ]:
pod_quality = con.execute('''
SELECT
    count(*) AS rows,
    approx_count_distinct(pod_id) AS approximate_unique_pods,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE workload_id = '') AS blank_workload_id,
    count(*) FILTER (WHERE avg_gpu_sm_util IS NULL) AS missing_avg_gpu_sm_util,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM pod_profile_raw
''').df()

job_quality = con.execute('''
SELECT
    count(*) AS rows,
    approx_count_distinct(pod_id) AS approximate_unique_pods,
    count(*) FILTER (WHERE workload_id IS NULL) AS missing_workload_id,
    count(*) FILTER (WHERE workload_id = '') AS blank_workload_id,
    count(*) FILTER (WHERE duration_hours IS NULL) AS missing_duration,
    count(*) FILTER (WHERE duration_hours < 0) AS negative_duration,
    count(*) FILTER (WHERE schedule_delay_sec IS NULL) AS missing_schedule_delay,
    count(*) FILTER (WHERE schedule_delay_sec < 0) AS negative_schedule_delay,
    count(*) FILTER (WHERE ready_delay_sec IS NULL) AS missing_ready_delay,
    count(*) FILTER (WHERE ready_delay_sec < 0) AS negative_ready_delay
FROM job_raw
''').df()

def quality_table(frame, dataset):
    row = frame.iloc[0]
    total = int(row["rows"])
    records = []
    for metric, value in row.items():
        records.append({
            "dataset": dataset,
            "metric": metric,
            "count": int(value),
            "percent_of_rows": 100.0 * float(value) / total,
        })
    return pd.DataFrame(records)

quality_report = pd.concat([
    quality_table(pod_quality, f"pod_hourly (days {POD_PROFILE_DAYS})"),
    quality_table(job_quality, "execution_summary"),
], ignore_index=True)
display(quality_report)


### Interpretation rules

- Missing values are not removed globally.
- Literal `Unknown` or `unknown` values remain valid source categories.
- Negative delays are preserved in raw columns but excluded from clean delay fields.
- GPU-utilization missingness should also be evaluated after filtering to `gpu_request > 0`; many pod-hour rows do not request a GPU.
- Extreme duration or utilization values are flagged for study, not automatically deleted.


## 2. Query-time adjustments

These views do not create another large processed file. They preserve every raw column and add clearly named analysis fields.

`workload_id` treatment is intentionally conservative because the team has not chosen a final strategy. Blank strings are normalized to `NULL`, but missing IDs are not imputed and rows are not dropped.


In [ ]:
con.execute('''
CREATE OR REPLACE VIEW pod_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean,
    gpu_request > 0 AS is_gpu_request
FROM pod_raw
''')

con.execute('''
CREATE OR REPLACE VIEW job_analysis AS
SELECT
    *,
    nullif(trim(workload_id), '') AS workload_id_clean,
    CASE
        WHEN workload_id IS NULL OR trim(workload_id) = '' THEN 'missing'
        ELSE 'observed'
    END AS workload_id_status,
    CASE WHEN schedule_delay_sec >= 0 THEN schedule_delay_sec END
        AS schedule_delay_sec_clean,
    CASE WHEN ready_delay_sec >= 0 THEN ready_delay_sec END
        AS ready_delay_sec_clean
FROM job_raw
''')

con.execute(f'''
CREATE OR REPLACE VIEW pod_profile_analysis AS
SELECT * FROM pod_analysis WHERE day IN ({profile_days_sql})
''')

preprocessing_policy = pd.DataFrame([
    ["workload_id", "Trim blanks to NULL; do not impute; keep all rows"],
    ["schedule_delay_sec", "Keep raw; add non-negative *_clean field"],
    ["ready_delay_sec", "Keep raw; add non-negative *_clean field"],
    ["Unknown categories", "Keep as documented categories"],
    ["numeric outliers", "Keep and profile before any research-specific rule"],
])
preprocessing_policy.columns = ["field", "current_rule"]
display(preprocessing_policy)


In [ ]:
workload_id_coverage = con.execute('''
SELECT
    'pod_hourly' AS dataset,
    count(*) AS rows,
    count(*) FILTER (WHERE workload_id_clean IS NOT NULL) AS rows_with_workload_id,
    approx_count_distinct(workload_id_clean) AS approximate_unique_workloads
FROM pod_profile_analysis
UNION ALL
SELECT
    'execution_summary',
    count(*),
    count(*) FILTER (WHERE workload_id_clean IS NOT NULL),
    approx_count_distinct(workload_id_clean)
FROM job_analysis
''').df()
workload_id_coverage["coverage_percent"] = (
    100 * workload_id_coverage["rows_with_workload_id"] / workload_id_coverage["rows"]
)
display(workload_id_coverage)


### Pending `workload_id` decision

Recommended default: keep missing IDs for single-table analyses. For a workload-level aggregation or join, use only non-missing `workload_id_clean` values and report the retained-row and retained-workload coverage.

Do not silently replace a missing workload ID with `pod_id`: that changes the analytical grain. If the team later needs a fallback grouping key, create a separate, explicitly named field and document its meaning.


## 3. Data profile tables

DuckDB's `SUMMARIZE` command returns one row per field with type, non-null count, null percentage, approximate unique count, range, average, standard deviation, and quartiles where applicable. Pod profiles use `POD_PROFILE_DAYS`; the smaller execution-summary file is profiled in full.


In [ ]:
pod_column_profile = con.execute(
    "SUMMARIZE SELECT * FROM pod_profile_analysis"
).df()
pod_column_profile.insert(0, "dataset", f"pod_hourly (days {POD_PROFILE_DAYS})")

job_column_profile = con.execute(
    "SUMMARIZE SELECT * FROM job_analysis"
).df()
job_column_profile.insert(0, "dataset", "execution_summary")

column_profile = pd.concat(
    [pod_column_profile, job_column_profile],
    ignore_index=True,
)
display(column_profile)


In [ ]:
def numeric_profile(table, dataset, columns):
    expressions = ["count(*) AS total_rows"]
    for column in columns:
        expressions.extend([
            f'count("{column}") AS "{column}__non_null"',
            f'min("{column}") AS "{column}__min"',
            f'approx_quantile("{column}", 0.25) AS "{column}__q25"',
            f'approx_quantile("{column}", 0.50) AS "{column}__median"',
            f'approx_quantile("{column}", 0.75) AS "{column}__q75"',
            f'approx_quantile("{column}", 0.99) AS "{column}__p99"',
            f'max("{column}") AS "{column}__max"',
            f'avg("{column}") AS "{column}__mean"',
            f'stddev_pop("{column}") AS "{column}__sd"',
        ])
    row = con.execute(
        f"SELECT {', '.join(expressions)} FROM {table}"
    ).df().iloc[0]
    total = int(row["total_rows"])
    records = []
    for column in columns:
        records.append({
            "dataset": dataset,
            "column": column,
            "non_null_count": int(row[f"{column}__non_null"]),
            "non_null_percent": 100 * float(row[f"{column}__non_null"]) / total,
            "min": row[f"{column}__min"],
            "q25": row[f"{column}__q25"],
            "median": row[f"{column}__median"],
            "q75": row[f"{column}__q75"],
            "p99": row[f"{column}__p99"],
            "max": row[f"{column}__max"],
            "mean": row[f"{column}__mean"],
            "sd": row[f"{column}__sd"],
        })
    return pd.DataFrame(records)

pod_numeric = numeric_profile("pod_profile_analysis", f"pod_hourly (days {POD_PROFILE_DAYS})", [
    "gpu_request", "used_gpu_hours", "avg_gpu_sm_util",
    "avg_gpu_mem_gib", "cpu_request_cores",
    "schedule_delay_sec_clean", "ready_delay_sec_clean",
])
job_numeric = numeric_profile("job_analysis", "execution_summary", [
    "gpu_request", "duration_hours",
    "schedule_delay_sec_clean", "ready_delay_sec_clean",
])
numeric_distribution = pd.concat([pod_numeric, job_numeric], ignore_index=True)
display(numeric_distribution)


In [ ]:
def top_categories(table, dataset, columns, top_n=10):
    total = con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
    frames = []
    for column in columns:
        frame = con.execute(f'''            SELECT
                '{dataset}' AS dataset,
                '{column}' AS column,
                coalesce(cast("{column}" AS VARCHAR), '<NULL>') AS value,
                count(*) AS count
            FROM {table}
            GROUP BY "{column}"
            ORDER BY count DESC
            LIMIT {int(top_n)}
        ''').df()
        frame["percent"] = 100 * frame["count"] / total
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

pod_categories = top_categories("pod_profile_analysis", f"pod_hourly (days {POD_PROFILE_DAYS})", [
    "state_public", "gpu_spec_public", "priority_class",
    "job_type_public", "model_type_public", "is_genai_request",
    "ready_status", "workload_id_status",
])
job_categories = top_categories("job_analysis", "execution_summary", [
    "gpu_spec_public", "priority_class", "job_type_public",
    "model_type_public", "is_genai_request", "ready_status",
    "schedule_status", "workload_id_status",
])
categorical_distribution = pd.concat(
    [pod_categories, job_categories],
    ignore_index=True,
)
display(categorical_distribution)


## Handoff notes

- All profile tables are ordinary pandas DataFrames and can be exported later if the team wants small CSV summaries.
- Keep filtering, cleaning, and aggregation lazy in DuckDB or Arrow; do not load either full dataset into pandas or R memory.
- Decide the final `workload_id` rule only after selecting the research question and reporting coverage.
- Aggregate each source to a declared grain before joining. Do not raw-join the two large tables.
